
# Tầng 1 – DFM backbone cho đề tài nowcasting GDP thời gian thực

Notebook này triển khai **tầng 1** trong pipeline của đề tài **“Real-Time GDP Growth Nowcasting using a Hybrid Dynamic Factor Model with Machine Learning Residual Correction”**.

Mục tiêu của notebook:

1. khóa protocol thực nghiệm theo đúng research design;
2. dựng `target_vintage_table`, `truth_third_release`, `truth_latest_rtdsm`, `truth_gdpplus`;
3. đọc đúng snapshot FRED-MD theo từng month-end vintage;
4. transform + standardize theo nguyên tắc **vintage-by-vintage, không leakage**;
5. fit **block-structured mixed-frequency DFM** bằng EM + Kalman;
6. sinh các artefact cần cho tầng ML sau này:
   - `dfm_nowcasts.csv`
   - `dfm_states.csv`
   - `dfm_news_series.csv`
   - `dfm_news_blocks.csv`
   - `dfm_coverage.csv`
   - `dfm_diagnostics.csv`
   - `vintage_manifest.csv`

Notebook được thiết kế để chạy trực tiếp trên repo dữ liệu local của bạn. Mặc định notebook chạy ở chế độ **debug** để kiểm tra pipeline trước; sau đó bạn chỉ cần đổi `RUN_MODE = "research"` để chạy toàn bộ cửa sổ nghiên cứu.



## Cấu trúc làm việc

Notebook này dùng một file hỗ trợ tên `dfm_layer1_support.py`.

**Đặt hai file sau vào root của repo** trước khi chạy:

- `dfm_layer1_backbone.ipynb`
- `dfm_layer1_support.py`

Repo của bạn đã có sẵn `hybrid_nowcast_utils.py`; notebook này tận dụng file đó để giữ đúng data architecture và bước làm trong đề cương.


In [ ]:

from __future__ import annotations

import json
import platform
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import dfm_layer1_support as dfs

print('Python   :', platform.python_version())
print('Pandas   :', pd.__version__)
try:
    import statsmodels
    print('Statsmodels:', statsmodels.__version__)
except Exception as exc:
    print('Statsmodels import error:', exc)
print('Project root:', PROJECT_ROOT)



## Bước 0 – Khóa protocol và dò đúng file local

Cell dưới đây sẽ:

- tìm `data/raw` trong repo;
- đọc `experiment_config.json` nếu có;
- dò đúng các file target/truth (`routputMvQd.xlsx`, `routput_first_second_third.xlsx`, `GDPplus_Vintages.xlsx`, `meanGrowth.xlsx`);
- dựng catalog FRED-MD từ các thư mục lịch sử + thư mục current monthly.


In [ ]:

files = dfs.discover_project_files(PROJECT_ROOT)
config = dfs.load_experiment_config(files.experiment_config)
fred_md_catalog = dfs.build_fred_md_catalog(files.fred_md_dirs)
target_vintage_table, truth_tables = dfs.build_target_and_truth_tables(files)
status_df = dfs.compute_data_status(target_vintage_table, truth_tables, fred_md_catalog)

print('Raw directory :', files.raw_dir)
print('Output dir    :', files.output_dir)
print('FRED-MD dirs  :')
for p in files.fred_md_dirs:
    print('  -', p)

print('
Detected key files:')
for label, path in {
    'experiment_config': files.experiment_config,
    'routput_monthly': files.routput_monthly,
    'routput_release_values': files.routput_release_values,
    'gdpplus_vintages': files.gdpplus_vintages,
    'spf_mean_growth': files.spf_mean_growth,
    'full_panel_map': files.full_panel_map,
}.items():
    print(f'  - {label}: {path}')

print('
Data status:')
display(status_df)



## Bước 1 – Kiểm tra target/truth tables

Ba bảng quan trọng phải tách riêng:

- `target_vintage_table`: GDP history theo **vintage**, dùng cho estimation;
- `truth_third_release`: truth chính để chấm điểm;
- `truth_latest_rtdsm` và `truth_gdpplus_latest`: robustness truths.


In [ ]:

print('target_vintage_table')
display(target_vintage_table.head())
print('truth_third_release')
display(truth_tables['truth_third_release'].head())
print('truth_latest_rtdsm')
display(truth_tables['truth_latest_rtdsm'].head())
if 'truth_gdpplus_latest' in truth_tables:
    print('truth_gdpplus_latest')
    display(truth_tables['truth_gdpplus_latest'].head())



## Bước 2 – Cấu hình run

- `RUN_MODE = "debug"`: chỉ chạy vài vintage gần nhất để kiểm tra toàn bộ pipeline;
- `RUN_MODE = "research"`: chạy toàn bộ benchmark window từ `2000Q1` đến **latest local predictor vintage**;
- `PANEL_MODE = "stable_subset"`: chạy được ngay và rất phù hợp để debug;
- `PANEL_MODE = "full_panel"`: chỉ nên dùng khi bạn đã hoàn thiện `series_to_block_full_panel.csv`.


In [ ]:

RUN_MODE = 'debug'          # 'debug' hoặc 'research'
PANEL_MODE = 'stable_subset'  # 'stable_subset' hoặc 'full_panel'
START_QUARTER = config.get('benchmark_window', {}).get('start_quarter', '2000Q1')
END_VINTAGE = None          # None => tự động lấy latest local vintage; ví dụ '2026-02'
DEBUG_N_VINTAGES = 6        # chỉ dùng khi RUN_MODE='debug'
DEBUG_RECENT = True         # True => lấy 6 vintage gần nhất
P_GRID = tuple(config.get('baseline_factor_design', {}).get('factor_lag_grid', [1, 2, 3]))
MAXITER = 40 if RUN_MODE == 'debug' else 200
TOLERANCE = 1e-6
DISP = False
AR_BENCHMARK_ORDER = 2

runtime_protocol = {
    'run_mode': RUN_MODE,
    'panel_mode': PANEL_MODE,
    'start_quarter': START_QUARTER,
    'end_vintage': str(END_VINTAGE) if END_VINTAGE is not None else None,
    'debug_n_vintages': DEBUG_N_VINTAGES,
    'debug_recent': DEBUG_RECENT,
    'p_grid': list(P_GRID),
    'maxiter': MAXITER,
    'tolerance': TOLERANCE,
    'ar_benchmark_order': AR_BENCHMARK_ORDER,
    'latest_predictor_vintage_detected': status_df.loc[status_df['object'] == 'latest_predictor_vintage', 'value'].iloc[0],
    'latest_main_truth_quarter_detected': status_df.loc[status_df['object'] == 'latest_main_truth_quarter', 'value'].iloc[0],
}

(files.output_dir / 'runtime_protocol.json').write_text(json.dumps(runtime_protocol, indent=2, ensure_ascii=False), encoding='utf-8')
runtime_protocol



## Bước 3–4 – Metadata stable subset và mapping full panel

- Stable subset dùng trực tiếp từ data dictionary và đã gắn block + anchor.
- Nếu bạn chọn `PANEL_MODE = "full_panel"` nhưng chưa có file mapping, cell dưới đây sẽ tự động sinh template để bạn điền block cho toàn bộ series.


In [ ]:

stable_meta = dfs.stable_metadata()
print('Stable subset block counts:')
display(stable_meta.groupby('block').size().rename('n_series').reset_index())

if PANEL_MODE == 'full_panel' and files.full_panel_map is None:
    ref_vintage = fred_md_catalog['vintage'].max()
    ref_snapshot, ref_tcodes, ref_path = dfs.load_local_fred_md_snapshot(ref_vintage, fred_md_catalog)
    template_path = files.output_dir / 'series_to_block_full_panel_template.csv'
    dfs.generate_full_panel_template(ref_snapshot.columns, stable_meta, template_path)
    raise FileNotFoundError(
        'Bạn đang chọn PANEL_MODE="full_panel" nhưng repo chưa có file mapping block đầy đủ. '        f'Notebook đã tạo template tại: {template_path}. Hãy điền block ex ante rồi chạy lại.'
    )

if files.full_panel_map is not None:
    print('Detected full-panel map:', files.full_panel_map)



## Bước 5 – Chọn các vintage sẽ chạy

`select_vintages_for_run` luôn lấy đúng **month-end vintages** từ local FRED-MD catalog. Nếu `END_VINTAGE=None`, notebook sẽ dùng vintage mới nhất hiện có trong thư mục local của bạn.


In [ ]:

vintages = dfs.select_vintages_for_run(
    fred_md_catalog=fred_md_catalog,
    start_quarter=START_QUARTER,
    end_vintage=END_VINTAGE,
    run_mode=RUN_MODE,
    debug_n_vintages=DEBUG_N_VINTAGES,
    debug_recent=DEBUG_RECENT,
)

print('Number of vintages selected:', len(vintages))
print('First vintage:', vintages[0] if vintages else None)
print('Last vintage :', vintages[-1] if vintages else None)
pd.DataFrame({'vintage': [str(v) for v in vintages[:10]]})



## Bước 6–11 – Chạy DFM backbone theo pseudo-real-time loop

Cell dưới đây sẽ làm đúng trình tự:

1. freeze snapshot tại vintage `v`;
2. đọc đúng FRED-MD snapshot local;
3. transform bằng official tcode;
4. standardize theo sample hiện có tại chính vintage đó;
5. fit DFM block-structured bằng EM + Kalman;
6. lấy nowcast current quarter;
7. tính news decomposition so với vintage trước;
8. lưu toàn bộ artefact vào `outputs/dfm_layer1/`.

> Lưu ý: với `RUN_MODE = "research"`, cell này sẽ chạy khá nặng. Hãy dùng debug trước để kiểm tra pipeline rồi mới chuyển sang research run.


In [ ]:

artifacts = dfs.run_dfm_backbone(
    vintages=vintages,
    target_vintage_table=target_vintage_table,
    fred_md_catalog=fred_md_catalog,
    stable_meta=stable_meta,
    output_dir=files.output_dir,
    panel_mode=PANEL_MODE,
    full_panel_map_path=files.full_panel_map,
    p_grid=P_GRID,
    maxiter=MAXITER,
    tolerance=TOLERANCE,
    disp=DISP,
    ar_benchmark_order=AR_BENCHMARK_ORDER,
)

print('Artifacts written to:', files.output_dir)
for name in sorted(artifacts):
    print(f'{name}: {artifacts[name].shape}')



## Bước 12 – Kiểm tra artefact đã sinh ra

Ba bảng quan trọng nhất để nhìn nhanh:

- `dfm_nowcasts`
- `dfm_diagnostics`
- `vintage_manifest`


In [ ]:

display(artifacts['dfm_nowcasts'].tail(10))
display(artifacts['dfm_diagnostics'].tail(10))
display(artifacts['vintage_manifest'].tail(10))



## Bước 13 – Validation trước khi sang tầng ML

Theo outline, trước khi sang residual-correction bạn phải kiểm tra tối thiểu:

1. EM có hội tụ ổn định không;
2. orientation của factor có ổn không;
3. coverage theo block có đầy đủ không;
4. news decomposition có đọc được không;
5. DFM-only có phải benchmark đủ mạnh so với AR(2) đơn giản không.

Cell dưới đây tạo một bảng kiểm tra nhanh.


In [ ]:

diagnostics = artifacts['dfm_diagnostics'].copy()
nowcasts = artifacts['dfm_nowcasts'].copy()
coverage = artifacts['dfm_coverage'].copy()
news_blocks = artifacts['dfm_news_blocks'].copy()

anchor_cols = [c for c in diagnostics.columns if c.startswith('anchor_ok_')]
anchor_ok = diagnostics[anchor_cols].fillna(False).all().all() if anchor_cols else False
coverage_ok = coverage.groupby('vintage')['block'].nunique().min() >= 6 if not coverage.empty else False
latest_processed_ok = str(vintages[-1]) == nowcasts['vintage'].max() if len(vintages) else False
news_nonempty = not news_blocks.empty

checklist = pd.DataFrame([
    {'check': 'Latest processed vintage equals requested end vintage', 'ok': latest_processed_ok},
    {'check': 'At least one block-news table exists', 'ok': news_nonempty},
    {'check': 'All represented anchor loadings are non-negative after sign orientation', 'ok': anchor_ok},
    {'check': 'Each processed vintage has 6 block coverage entries', 'ok': coverage_ok},
    {'check': 'Required CSV artefacts exist in outputs/dfm_layer1', 'ok': all((files.output_dir / f).exists() for f in [
        'dfm_nowcasts.csv',
        'dfm_states.csv',
        'dfm_news_series.csv',
        'dfm_news_blocks.csv',
        'dfm_coverage.csv',
        'dfm_diagnostics.csv',
        'vintage_manifest.csv',
    ])},
])

display(checklist)



## Bước 14 – Scoring sơ bộ DFM-only so với AR(2)

Đây chưa phải evaluation đầy đủ của paper, nhưng đủ để kiểm tra rằng backbone DFM không quá yếu trước khi ghép ML.

- Truth dùng ở đây là **third-release**.
- Chỉ những quý đã có third-release mới được chấm điểm.


In [ ]:

scored = dfs.attach_truth_and_score(
    nowcasts=artifacts['dfm_nowcasts'],
    truth_table=truth_tables['truth_third_release'],
    truth_col='truth_third_release',
)
validation_summary = dfs.pre_ml_validation_summary(scored, truth_col='truth_third_release')

print('Scored observations:', scored['truth_third_release'].notna().sum())
display(validation_summary)


In [ ]:

print('Recent nowcast revisions:')
display(
    artifacts['dfm_nowcasts'][[
        'vintage', 'quarter', 'tau', 'dfm_nowcast_ann_pct',
        'revision_from_previous_vintage_ann_pct', 'ar2_benchmark_ann_pct'
    ]].tail(12)
)

if not artifacts['dfm_news_blocks'].empty:
    print('Latest block-level news:')
    latest_v = artifacts['dfm_news_blocks']['current_vintage'].max()
    display(
        artifacts['dfm_news_blocks']
        .loc[artifacts['dfm_news_blocks']['current_vintage'] == latest_v]
        .sort_values(['quarter', 'block'])
        .tail(12)
    )

print('Latest block coverage:')
latest_cov_v = artifacts['dfm_coverage']['vintage'].max()
display(
    artifacts['dfm_coverage']
    .loc[artifacts['dfm_coverage']['vintage'] == latest_cov_v]
    .sort_values('block')
)



## Những file cần kiểm tra để biết tầng 1 đã hoàn thành

Nếu notebook chạy xong và **7 file sau** đã xuất hiện trong `outputs/dfm_layer1/`, thì tầng 1 đã hoàn thành đúng nghĩa “feature-generating econometric engine”:

1. `dfm_nowcasts.csv` – nowcast theo từng vintage / quarter / tau;
2. `dfm_states.csv` – trạng thái factor đã smoothed cho từng vintage;
3. `dfm_news_series.csv` – news theo series;
4. `dfm_news_blocks.csv` – signed/absolute block news;
5. `dfm_coverage.csv` – coverage theo block;
6. `dfm_diagnostics.csv` – hội tụ, lag order, anchor checks;
7. `vintage_manifest.csv` – manifest để audit từng vintage đã chạy.

### Cách kết luận tầng 1 đã “đủ sạch”

Bạn nên xác nhận đồng thời 4 ý sau:

- `vintage_manifest.csv` chạy hết đến **latest local predictor vintage** mà bạn mong muốn (ví dụ `2026-02` nếu local data của bạn đã cập nhật đến đó);
- `dfm_diagnostics.csv` không cho thấy lỗi fit hàng loạt hoặc anchor orientation bị sai;
- `dfm_news_blocks.csv` có block news đọc được và không rỗng từ vintage thứ hai trở đi;
- `dfm_nowcasts.csv` cho thấy DFM-only ít nhất không yếu hơn benchmark AR(2) trong validation sơ bộ.

Khi 4 điều này đều đạt, bạn có thể chuyển sang xây tầng 2: **ML residual correction**.
